# NExT-GQA Answerer — Qwen2-VL-7B
Chạy answerer dùng **Qwen2-VL-7B-Instruct** trên Google Colab (T4 hoặc A100).

**Checklist trước khi chạy:**
- Runtime → Change runtime type → GPU (T4 hoặc A100)
- Google Drive đã có thư mục `videos` (cấu trúc VidOR)
- Repo đã push `answerer_qwen.py`, `eval_qa.py`, `VideoMind`


## 1) Kiểm tra GPU


In [ ]:
!nvidia-smi


## 2) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3) Clone repo


In [ ]:
import os
import subprocess

# TODO: đổi lại URL repo của bạn
REPO_URL = 'https://github.com/YOUR_USERNAME/TFVTG.git'
REPO_DIR = '/content/TFVTG'

def sh(cmd: str):
    print('+', cmd)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.exists(REPO_DIR):
    sh(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('Repo already cloned, pulling latest...')
    sh(f'git -C {REPO_DIR} pull')

%cd {REPO_DIR}
!ls .


## 4) Cấu hình đường dẫn
Chỉ cần chỉnh **2 biến** bên dưới.


In [ ]:
# Thư mục VidOR videos trên Drive (cùng cấu trúc với feature extraction)
# Ví dụ: .../nextqa/videos/0034/4740345442.mp4
DRIVE_VIDOR_ROOT = '/content/drive/MyDrive/KLTN/VideoMind-Dataset/nextqa/videos'

# Thư mục lưu answerer outputs trên Drive — để không mất khi session reset
DRIVE_OUTPUTS_DIR = '/content/drive/MyDrive/KLTN/nextgqa_answerer_outputs'

print('VIDOR_ROOT :', DRIVE_VIDOR_ROOT)
print('OUTPUTS_DIR:', DRIVE_OUTPUTS_DIR)


## 5) Symlink videos + outputs vào repo


In [ ]:
import os

os.makedirs(DRIVE_OUTPUTS_DIR, exist_ok=True)

nextgqa_dir = f'{REPO_DIR}/dataset/nextgqa'
os.makedirs(nextgqa_dir, exist_ok=True)

# Symlink videos → không copy, quá nặng
videos_link = f'{nextgqa_dir}/videos'
if not os.path.exists(videos_link):
    os.symlink(DRIVE_VIDOR_ROOT, videos_link)
    print(f'Symlinked videos: {videos_link} -> {DRIVE_VIDOR_ROOT}')
else:
    print('videos symlink already exists')

# Symlink outputs → lưu thẳng vào Drive, tránh mất khi session reset
outputs_link = f'{nextgqa_dir}/qwen_outputs'
if not os.path.exists(outputs_link):
    os.symlink(DRIVE_OUTPUTS_DIR, outputs_link)
    print(f'Symlinked outputs: {outputs_link} -> {DRIVE_OUTPUTS_DIR}')
else:
    print('outputs symlink already exists')

print('\ndataset/nextgqa/ contents:')
!ls -la {nextgqa_dir}


## 6) Install dependencies
⚠️ Không cài lại torch / torchvision (Colab đã có sẵn). Chỉ cài các gói còn thiếu.


In [ ]:
# 1) Upgrade transformers (cần >= 4.45 cho Qwen2-VL)
# --no-deps tránh kéo lại torch
!pip install -q "transformers>=4.45,<5.0" --no-deps
!pip install -q "tokenizers>=0.19" --prefer-binary

# 2) Các gói cần thiết chưa có trong Colab
!pip install -q accelerate peft safetensors

# 3) decord (video decoding cho VideoMind process_vision_info)
!pip install -q decord --prefer-binary

# 4) bitsandbytes (cho --load_in_4bit trên T4)
!pip install -q bitsandbytes

# 5) VideoMind deps nhẹ
!pip install -q nncore tabulate

# 6) huggingface_hub (nếu muốn snapshot_download)
!pip install -q huggingface_hub

# 7) Cài VideoMind package (chỉ register dataset classes + utils, không pull torch)
!pip install -q --no-deps -e VideoMind/

print('\nVerify imports...')
import torch, transformers, decord
print(' torch        :', torch.__version__)
print(' transformers :', transformers.__version__)
print(' decord       :', getattr(decord, '__version__', 'unknown'))
print(' CUDA available:', torch.cuda.is_available())


## 7) Cấu hình model


In [ ]:
import os
import torch

# Option A: download từ HuggingFace cache (tự động, ~16GB, mất 5-10 phút lần đầu)
# MODEL_PATH = 'Qwen/Qwen2-VL-7B-Instruct'

# Option B: lưu model vào Drive để tái sử dụng (khuyến nghị nếu chạy nhiều lần)
from huggingface_hub import snapshot_download

MODEL_PATH = '/content/drive/MyDrive/KLTN/models/Qwen2-VL-7B-Instruct'
if not os.path.isdir(MODEL_PATH):
    print('Downloading model to Drive...')
    snapshot_download(repo_id='Qwen/Qwen2-VL-7B-Instruct', local_dir=MODEL_PATH)
    print('Done.')

# Mặc định fp16 (giống VideoMind), không cần quantization trên A100/L4
# Nếu dùng T4 (15GB) thì thêm --load_in_4bit khi chạy ở bước 8
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU VRAM: {vram_gb:.1f} GB')
print('Model:', MODEL_PATH)


## 8) Chạy Answerer
Sẽ tiếp tục từ checkpoint nếu đã chạy một phần — chạy lại cell này là đủ khi session bị ngắt.


In [ ]:
# ---- Configuration ----
SPLIT = 'test'      # 'test' or 'val'
MAX_FRAMES = 32     # frames mỗi clip
SAVE_EVERY = 50     # lưu checkpoint mỗi N câu hỏi
FIRST_N = None      # đặt số nhỏ (VD: 10) để test nhanh trước khi chạy full

OUTPUT_PATH = f'dataset/nextgqa/qwen_outputs/answerer_outputs_qwen_{SPLIT}.json'

cmd = (
    f'python -m nextgqa.answerer_qwen'
    f' --split {SPLIT}'
    f' --model_path {MODEL_PATH}'
    f' --output {OUTPUT_PATH}'
    f' --max_frames {MAX_FRAMES}'
    f' --save_every {SAVE_EVERY}'
)
if FIRST_N:
    cmd += f' --first_n {FIRST_N}'
# Thêm --load_in_4bit nếu dùng T4 (15GB VRAM)
# cmd += ' --load_in_4bit'

print('Command:', cmd)
!{cmd}


## 9) Evaluate kết quả


In [ ]:
cmd_eval = (
    f'python -m nextgqa.eval_qa'
    f' --split {SPLIT}'
    f' --source qwen'
    f' --input {OUTPUT_PATH}'
)
print('Command:', cmd_eval)
!{cmd_eval}


## 10) So sánh Gemini vs Qwen2-VL (optional)


In [ ]:
import json, os

def load_answered(path):
    if not os.path.exists(path):
        print(f'Not found: {path}')
        return []
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    return [e for e in data if 'predicted_answer' in e]

gemini_path = 'dataset/nextgqa/answerer_outputs_test.json'
qwen_path = f'dataset/nextgqa/qwen_outputs/answerer_outputs_qwen_{SPLIT}.json'

for label, path in [('Gemini', gemini_path), ('Qwen2-VL', qwen_path)]:
    entries = load_answered(path)
    if not entries:
        continue
    acc = sum(1 for e in entries if e.get('predicted_answer_idx') == e.get('answer_idx')) / len(entries)
    print(f'{label} Acc@QA: {acc*100:.2f}% ({len(entries)} answered)')
